# Clinical NLP Pipeline — Google Colab Runner

**Predicting 30-Day Hospital Readmission from Discharge Notes**

This notebook runs the full pipeline on Google Colab. It mirrors `notebooks/full_pipeline.ipynb` but:
- Clones the repo from GitHub
- Installs all dependencies
- Uses `spaCy nlp.pipe()` with batching + multiprocessing to avoid the slow per-document tokenization loop
- Optionally mounts Google Drive to persist results

> **Runtime note:** This pipeline is **CPU-bound** (spaCy, NLTK, gensim, scikit-learn). A GPU runtime will **not** speed it up. Use `Runtime → Change runtime type → CPU` (or High-RAM CPU if available).

## 0. Environment Setup

Clone the repo and install dependencies. This takes ~3–5 minutes on a fresh Colab instance.

In [ ]:
# Clone the project repo
import os, sys
REPO_URL = "https://github.com/IamPrashantBhattarai/Clinical-NLP.git"
REPO_DIR = "/content/Clinical-NLP"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

In [ ]:
# Install Python dependencies
# Colab already has many of these; -q keeps output short
!pip install -q -r requirements.txt

In [ ]:
# Download spaCy + NLTK models
!python -m spacy download en_core_web_sm -q

import nltk
for res in ["stopwords", "wordnet", "punkt", "punkt_tab", "averaged_perceptron_tagger"]:
    nltk.download(res, quiet=True)
print("Models ready.")

In [ ]:
# Wire up imports and logging
import sys, logging, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("colab_pipeline")
logger.info("Environment ready.")

## 0b. (Optional) Mount Google Drive to persist results

Colab VMs are ephemeral — everything in `/content` is wiped when the session ends. Mount Drive if you want figures, models, and processed data saved permanently.

In [ ]:
SAVE_TO_DRIVE = True  # set to False to skip

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = Path("/content/drive/MyDrive/ClinicalNLP_results")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Results will be copied to:", DRIVE_OUT)
else:
    DRIVE_OUT = None

## 1. Data Generation & Loading

Generate synthetic patients, admissions, and discharge notes, then load them into DataFrames.

In [ ]:
from src.generate_synthetic_data import run as generate_data
from src.data_loader import load_all

generate_data()
data = load_all()
print({k: len(v) for k, v in data.items()})
data["notes"].head(2)

## 2. Preprocessing (optimized with `nlp.pipe`)

The repo's `build_preprocessing_pipeline` calls spaCy one doc at a time via `.apply`, which is the main bottleneck you hit locally. This cell **monkey-patches** `tokenize_clinical` path by overriding the tokenization step to use `nlp.pipe(batch_size=64, n_process=2)` — typically 5–20× faster on the same hardware. Source files on disk are left untouched.

In [ ]:
import string, re
import spacy
import pandas as pd
from src import preprocess as pp
from nltk.corpus import stopwords as nltk_stopwords

# Load config
import yaml
CFG_PATH = Path("config/config.yaml")
config = yaml.safe_load(CFG_PATH.read_text()) if CFG_PATH.exists() else {}
pp_cfg = config.get("preprocessing", {})

# Prepare stopwords identically to preprocess.tokenize_clinical
stop_words = set(nltk_stopwords.words("english"))
stop_words |= pp.DEFAULT_MEDICAL_STOPWORDS
stop_words |= set(pp_cfg.get("custom_stopwords", []))

# Load spaCy once and disable unused components for speed
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
RE_PURE_NUMBER = re.compile(r"^\d+\.?\d*$")

def _filter_doc(doc):
    out = []
    for tok in doc:
        lemma = tok.lemma_.lower().strip()
        if (tok.is_stop or tok.is_punct or tok.is_space
            or len(lemma) <= 1
            or lemma in stop_words
            or RE_PURE_NUMBER.match(lemma)
            or lemma in string.punctuation):
            continue
        out.append(lemma)
    return out

def fast_tokenize(texts, batch_size=64, n_process=2):
    return [_filter_doc(doc) for doc in nlp.pipe(texts, batch_size=batch_size, n_process=n_process)]

# Replicate build_preprocessing_pipeline, but swap the tokenization step
notes = data["notes"].copy()
text_col = "text"

# 1. Remove unwanted sections
sections_to_remove = pp_cfg.get("remove_sections", [])
if sections_to_remove:
    notes[text_col] = notes[text_col].apply(lambda t: pp.remove_sections(str(t), sections_to_remove))

# 2. Clean text
notes["cleaned_text"] = notes[text_col].apply(pp.clean_clinical_text)

# 3. Length filter
min_len = pp_cfg.get("min_note_length", 100)
max_len = pp_cfg.get("max_note_length", 50000)
before = len(notes)
notes = notes[notes["cleaned_text"].str.len().between(min_len, max_len)].reset_index(drop=True)
print(f"Filtered {before - len(notes)} notes outside length bounds — {len(notes)} remaining")

# 4. Batched tokenization  (the speed win)
%time notes["tokens"] = fast_tokenize(notes["cleaned_text"].tolist(), batch_size=64, n_process=2)

# 5. Bigrams / trigrams
notes["tokens"] = pp.create_bigrams_trigrams(notes["tokens"].tolist())
notes["num_tokens"] = notes["tokens"].apply(len)

vocab = {tok for toks in notes["tokens"] for tok in toks}
print(f"Vocab size: {len(vocab):,}  |  Avg tokens/doc: {notes['num_tokens'].mean():.1f}")

# Expose as `processed` so downstream stages match the original notebook's variable name
processed = notes
processed.head(2)

## 3. Topic Modeling (LDA)

Discover latent clinical topics in discharge notes.

In [ ]:
from src.topic_model import run_lda_pipeline

eligible = processed[processed["num_tokens"] >= 20].reset_index(drop=True)
lda_results = run_lda_pipeline(eligible, config=config)
lda_results["topics_df"]

## 4. Feature Engineering

Build TF-IDF, topic, structured, and combined feature sets.

In [ ]:
from src.feature_engineer import build_feature_sets

feature_sets = build_feature_sets(
    processed_df=processed,
    lda_results=lda_results,
    patients=data["patients"],
    admissions=data["admissions"],
    config=config,
)
print({k: v["X"].shape for k, v in feature_sets.items() if "X" in v})

## 5. Prediction Pipeline

Train Logistic Regression, Random Forest, XGBoost, and an ensemble.

In [ ]:
from src.predict import run_prediction_pipeline

prediction_results = run_prediction_pipeline(feature_sets, config=config)
best = prediction_results["best"]
print(f"Best model: {best['model_name']} on {best['feature_set']}")
print(f"AUROC: {best['metrics']['auroc']:.3f}  |  AUPRC: {best['metrics']['auprc']:.3f}")

## 6. Fairness Audit

In [ ]:
from src.fairness import run_fairness_audit

fairness_results = run_fairness_audit(
    prediction_results=prediction_results,
    patients=data["patients"],
    admissions=data["admissions"],
    config=config,
)
if fairness_results:
    display(fairness_results["summary_df"])

## 7. Visualization

In [ ]:
from src.visualize import generate_all_figures

# Attach labels for LDA topic-vs-readmission plots (matches the local notebook)
lda_results["labels"] = feature_sets["combined"]["y"] if "combined" in feature_sets else None

generate_all_figures(
    processed=processed,
    lda_results=lda_results,
    prediction_results=prediction_results,
    fairness_results=fairness_results,
    patients=data["patients"],
    admissions=data["admissions"],
    config=config,
)

In [ ]:
# Display the headline figures inline
from IPython.display import Image, display
for fig in ["roc_curves.png", "pr_curves.png", "confusion_matrices.png", "fairness_disparity.png"]:
    p = Path("results/figures") / fig
    if p.exists():
        print(fig)
        display(Image(str(p)))

## 8. Persist Results to Drive

Copy figures, trained models, and processed data out of the ephemeral Colab VM.

In [ ]:
import shutil, joblib

if DRIVE_OUT is not None:
    # Figures
    fig_src = Path("results/figures")
    if fig_src.exists():
        shutil.copytree(fig_src, DRIVE_OUT / "figures", dirs_exist_ok=True)

    # Save best model + vectorizer
    models_out = DRIVE_OUT / "models"
    models_out.mkdir(exist_ok=True)
    try:
        joblib.dump(best["model"], models_out / f"{best['model_name']}_{best['feature_set']}.joblib")
        print("Saved best model →", models_out)
    except Exception as e:
        print("Could not pickle best model:", e)

    # Save processed notes parquet
    try:
        processed.to_parquet(DRIVE_OUT / "processed_notes.parquet")
        print("Saved processed notes parquet")
    except Exception as e:
        processed.to_csv(DRIVE_OUT / "processed_notes.csv", index=False)
        print("Parquet failed, saved CSV instead:", e)

    print("\nAll results in:", DRIVE_OUT)
else:
    print("SAVE_TO_DRIVE is False — results remain in /content only and will be lost when the VM stops.")

---

## Summary

| Stage | Output |
|-------|--------|
| Data  | Synthetic patients / admissions / discharge notes |
| Preprocessing | Cleaned + tokenized (batched `nlp.pipe`) |
| Topic model | LDA topics + per-doc topic distributions |
| Features | TF-IDF, topic, structured, combined |
| Models | Logistic Regression, Random Forest, XGBoost, Ensemble |
| Fairness | Group metrics + disparity audit |
| Figures | `results/figures/*.png` (copied to Drive) |

**Remember:** This workload is CPU-bound. If you want it even faster, bump `n_process` in the preprocessing cell (Colab free tier usually has 2 vCPUs; Colab Pro gives you more).